# 04 · RAG agéntico (Agente + LLM + MCP + tool)

**Bloque:** 1:10–1:35 · **Práctica:** 1:19–1:35

### Teoría breve
- **RAG (Retrieval-Augmented Generation / Generación Aumentada por Recuperación):** recupera contexto relevante **antes** de responder.
- **Pipeline:** documentos → *chunks* (fragmentos) → *embeddings* → *vector store* → *retriever* → LLM.

### Lo que haremos distinto aquí
En vez de escribir el RAG "a mano" en el notebook, lo empaquetamos como una **herramienta MCP** (`buscar_documentos`) y dejamos que un **Agente** decida cuándo usarla.

Así combinamos **todo lo aprendido**:
**LLM** (el modelo) + **Agente** (decide) + **MCP** (protocolo) + **tool MCP** (que hace la recuperación) → **RAG**.

In [ ]:
import sys, os
REPO = os.path.abspath("..")
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from config import get_chat_model, get_embeddings
print("Entorno cargado ✔")

## Paso 1 · La recuperación (RAG) vive dentro de una tool MCP
Usamos un **servidor MCP dedicado a RAG** en `mcp_server/rag_server.py`. Ahí está el pipeline RAG completo:
trocea `data/historia_zelanor.md`, genera *embeddings*, construye el índice **FAISS** y devuelve los 3 fragmentos más relevantes.

> **Nota:** el documento es una **historia inventada** (la ciudad flotante de *Miravalle*). Así demostramos que las respuestas correctas vienen del RAG y **no** del preentrenamiento del modelo.

In [ ]:
print(open(os.path.join(REPO, "mcp_server", "rag_server.py"), encoding="utf-8").read())

## Paso 2 · Conectar el cliente MCP y tomar la tool de RAG
Lanzamos el servidor de RAG por `stdio` y nos quedamos con la herramienta `buscar_documentos`.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "rag": {
        "command": sys.executable,
        "args": [os.path.join(REPO, "mcp_server", "rag_server.py")],
        "transport": "stdio",
    }
})

tools = await client.get_tools()
rag_tool = next(t for t in tools if t.name == "buscar_documentos")
print("Tools disponibles:", [t.name for t in tools])
print("Usaremos:", rag_tool.name)

## Paso 3 · Probar la recuperación en vivo (el paso *retrieve*)
Llamamos a la tool directamente para ver qué fragmentos devuelve (aún **sin** el LLM).

In [ ]:
fragmentos = await rag_tool.ainvoke({"pregunta": "¿Quién fundó Miravalle y en qué año?"})
print(fragmentos)

## Paso 4 · El agente que hace RAG
Creamos un **agente** con el **LLM** y la **tool MCP**. El *system prompt* le indica que
**siempre busque** en la base de conocimiento antes de responder (RAG agéntico).

In [ ]:
from langchain.agents import create_agent

agente = create_agent(get_chat_model(), tools=[rag_tool],
    system_prompt=(
        "Eres una asistente experta en IA. Antes de responder SIEMPRE usa la "
        "herramienta 'buscar_documentos' para recuperar contexto. Responde usando "
        "SOLO esa información; si no está en el contexto, di que no lo sabes."
    ))

r = await agente.ainvoke({"messages": [
    {"role": "user", "content": "¿Quién lidera los Sombra-corsarios y por qué odia a Sorenia Valdés?"}
]})
print(r["messages"][-1].content)

## ✅ Checkpoint 4
1. Mira el **rastro de mensajes**: debe verse que el agente **llamó a `buscar_documentos`** antes de responder.
2. Haz una pregunta **fuera del documento** y confirma que el agente dice que **no lo sabe**.

In [ ]:
print("=== Rastro: el agente decidió usar la tool de RAG ===")
for m in r["messages"]:
    m.pretty_print()

print("\n=== Pregunta fuera del documento ===")
r2 = await agente.ainvoke({"messages": [
    {"role": "user", "content": "¿Quién ganó el mundial de fútbol de 1998?"}
]})
print(r2["messages"][-1].content)